# Prepare Calendar Table

Creates a standard date dimension table `dbx_caresync.silver.calendar` spanning **1940-01-01** to **2030-12-31** with key calendar attributes.

In [0]:
CREATE OR REPLACE TABLE dbx_caresync.silver.calendar
AS
WITH date_spine AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  -- === Surrogate Key ===
  CAST(date_format(date, 'yyyyMMdd') AS INT)        AS date_key,

  -- === Core Date ===
  date                                               AS date,

  -- === Day Attributes ===
  day(date)                                          AS day_of_month,
  dayofweek(date)                                    AS day_of_week,
  date_format(date, 'EEEE')                          AS day_name,
  date_format(date, 'EEE')                           AS day_name_short,
  dayofyear(date)                                    AS day_of_year,

  -- === Week Attributes ===
  weekofyear(date)                                   AS iso_week_of_year,

  -- === Month Attributes ===
  month(date)                                        AS month_number,
  date_format(date, 'MMMM')                          AS month_name,
  date_format(date, 'MMM')                            AS month_name_short,
  day(last_day(date))                                AS days_in_month,

  -- === Quarter Attributes ===
  quarter(date)                                      AS quarter_number,
  CONCAT('Q', quarter(date))                         AS quarter_name,

  -- === Year Attributes ===
  year(date)                                         AS year,
  date_format(date, 'yyyy-MM')                       AS year_month,
  CONCAT(year(date), '-Q', quarter(date))            AS year_quarter,
  CASE
    WHEN (year(date) % 4 = 0 AND year(date) % 100 != 0)
      OR year(date) % 400 = 0 THEN TRUE
    ELSE FALSE
  END                                                AS is_leap_year,

  -- === Weekend / Weekday Flag ===
  CASE
    WHEN dayofweek(date) IN (1, 7) THEN TRUE
    ELSE FALSE
  END                                                AS is_weekend,

  -- === Period Boundaries ===
  trunc(date, 'MM')                                  AS first_day_of_month,
  last_day(date)                                     AS last_day_of_month,
  trunc(date, 'quarter')                             AS first_day_of_quarter,
  last_day(add_months(trunc(date, 'quarter'), 2))    AS last_day_of_quarter,
  trunc(date, 'year')                                AS first_day_of_year,
  DATE(CONCAT(year(date), '-12-31'))                 AS last_day_of_year

FROM date_spine
ORDER BY date;

In [0]:
SELECT
  count(*)       AS total_rows,
  min(date)      AS min_date,
  max(date)      AS max_date,
  count(DISTINCT year) AS distinct_years
FROM dbx_caresync.silver.calendar;

In [0]:
SELECT * FROM dbx_caresync.silver.calendar LIMIT 10;